<a href="https://colab.research.google.com/github/Sh3eyaSom4anshi/-Mental-Health-and-music-Analysis-Python-EDA-Project-/blob/main/DM_CP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests pandas numpy

In [ ]:
#loading api
from google.colab import userdata

TMDB_TOKEN = userdata.get('TMDB_BEARER_TOKEN')

if not TMDB_TOKEN:
    raise ValueError("TMDB token not found. Check Colab Secrets.")

print("TMDB token loaded successfully.")

TMDB token loaded successfully.


In [ ]:
#checking the api works correctly ?
import requests

BASE_URL = "https://api.themoviedb.org/3"

# For a v4 Bearer Token, pass it in the Authorization header
url = f"{BASE_URL}/discover/movie"

params = {
    "language": "en-US",
    "sort_by": "popularity.desc",
    "include_adult": False,
    "page": 1
}

headers = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

print("Status code:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("Total movies matching query:", data["total_results"])
    print("Movies returned:", len(data["results"]))
    print("First movie:", data["results"][0]["title"])
else:
    print(response.text)
    response.raise_for_status()

Status code: 200
Total movies matching query: 20001
Movies returned: 20
First movie: Spider-Man: Brand New Day


In [ ]:
#fetching the movie from multiple film industry
import time
import pandas as pd
import requests

BASE_URL = "https://api.themoviedb.org/3"

# Pass the v4 Bearer Token in the headers
headers = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
}

# Original language codes for different film industries
languages = {
    "en": "English",
    "hi": "Hindi",
    "ko": "Korean",
    "mr": "Marathi",
    "te": "Telugu",
    "ta": "Tamil",
    "ml": "Malayalam",
    "kn": "Kannada",
    "bn": "Bengali",
    "ja": "Japanese"
}

all_movies = []

# Start small: 3 pages per language, 20 results per page
pages_per_language = 3

for language_code, language_name in languages.items():
    print(f"Fetching {language_name} movies...")

    for page in range(1, pages_per_language + 1):
        params = {
            "language": "en-US",
            "with_original_language": language_code,
            "sort_by": "popularity.desc",
            "include_adult": False,
            "page": page
        }

        response = requests.get(
            f"{BASE_URL}/discover/movie",
            headers=headers,
            params=params,
            timeout=30
        )

        response.raise_for_status()
        results = response.json()["results"]

        for movie in results:
            all_movies.append({
                "tmdb_id": movie.get("id"),
                "title": movie.get("title"),
                "original_title": movie.get("original_title"),
                "original_language": movie.get("original_language"),
                "release_date": movie.get("release_date"),
                "overview": movie.get("overview"),
                "genre_ids": movie.get("genre_ids"),
                "vote_average": movie.get("vote_average"),
                "vote_count": movie.get("vote_count"),
                "popularity": movie.get("popularity"),
                "poster_path": movie.get("poster_path"),
                "adult": movie.get("adult"),
                "original_language_name": language_name
            })

        time.sleep(0.15)

movies_df = pd.DataFrame(all_movies)

# Remove duplicate movies using their unique TMDB IDs
movies_df = movies_df.drop_duplicates(
    subset="tmdb_id"
).reset_index(drop=True)

print("\nData fetching completed.")
print("Unique movies fetched:", len(movies_df))
print("Dataset shape:", movies_df.shape)

display(movies_df.head())

Fetching English movies...
Fetching Hindi movies...
Fetching Korean movies...
Fetching Marathi movies...
Fetching Telugu movies...
Fetching Tamil movies...
Fetching Malayalam movies...
Fetching Kannada movies...
Fetching Bengali movies...
Fetching Japanese movies...

Data fetching completed.
Unique movies fetched: 600
Dataset shape: (600, 13)


,tmdb_id,title,original_title,original_language,release_date,overview,genre_ids,vote_average,vote_count,popularity,poster_path,adult,original_language_name
0,969681,Spider-Man: Brand New Day,Spider-Man: Brand New Day,en,2026-07-29,Fighting crime full-time as Spider-Man in a wo...,"[878, 28, 12]",8.400,4936,986.0191,/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg,False,English
1,1423191,Resident Evil,Resident Evil,en,2026-09-16,Medical courier Bryan unwittingly finds himsel...,"[27, 878, 12]",7.300,859,704.1066,/i7UyjfPio0VFHB9rBUZSFyhOoM8.jpg,False,English
2,1291595,Insidious: Out of the Further,Insidious: Out of the Further,en,2026-08-19,"Gemma, a young mother raising her daughter in ...","[27, 53]",7.600,568,454.8876,/4tTrW9dXCByS5wt2pXVWb58zNjz.jpg,False,English
3,1368337,The Odyssey,The Odyssey,en,2026-07-15,"Odysseus, the legendary King of Ithaca, embark...","[12, 28, 14]",8.011,4085,389.5944,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,False,English
4,1228834,The Fix,The Fix,en,2026-09-10,Disillusioned by the end of the war in Afghani...,"[28, 53]",6.773,55,276.9325,/yopXjun3ICFfJci2ukcEzceZjUs.jpg,False,English


In [ ]:
print("Dataset dimensions:", movies_df.shape)

print("\nColumn names:")
print(movies_df.columns.tolist())

print("\nMissing values:")
print(movies_df.isnull().sum())

print("\nMovies by original language:")
print(movies_df["original_language"].value_counts())

# Save the raw API data
movies_df.to_csv(
    "/content/tmdb_movies_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved: /content/tmdb_movies_raw.csv")

Dataset dimensions: (600, 13)

Column names:
['tmdb_id', 'title', 'original_title', 'original_language', 'release_date', 'overview', 'genre_ids', 'vote_average', 'vote_count', 'popularity', 'poster_path', 'adult', 'original_language_name']

Missing values:
tmdb_id                    0
title                      0
original_title             0
original_language          0
release_date               0
overview                   0
genre_ids                  0
vote_average               0
vote_count                 0
popularity                 0
poster_path               53
adult                      0
original_language_name     0
dtype: int64

Movies by original language:
original_language
en    60
hi    60
ko    60
mr    60
te    60
ta    60
ml    60
kn    60
bn    60
ja    60
Name: count, dtype: int64

Saved: /content/tmdb_movies_raw.csv


In [ ]:
#enrich the dataset using tmdb

import requests
import pandas as pd
import time
import json
from pathlib import Path

BASE_URL = "https://api.themoviedb.org/3"

headers = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
}

# Get the genre ID-to-name mapping
response = requests.get(
    f"{BASE_URL}/genre/movie/list",
    headers=headers,
    params={"language": "en-US"},
    timeout=30
)
response.raise_for_status()

genre_map = {
    genre["id"]: genre["name"]
    for genre in response.json()["genres"]
}

def get_movie_metadata(tmdb_id):
    """Fetch credits and keywords for one movie."""
    result = {
        "cast": [],
        "director": None,
        "keywords": []
    }

    # Credits: actors and director
    credits_response = requests.get(
        f"{BASE_URL}/movie/{int(tmdb_id)}/credits",
        headers=headers,
        timeout=30
    )
    credits_response.raise_for_status()
    credits = credits_response.json()

    result["cast"] = [
        person["name"]
        for person in credits.get("cast", [])[:10]
    ]

    result["director"] = next(
        (
            person["name"]
            for person in credits.get("crew", [])
            if person.get("job") == "Director"
        ),
        None
    )

    # Keywords
    keywords_response = requests.get(
        f"{BASE_URL}/movie/{int(tmdb_id)}/keywords",
        headers=headers,
        timeout=30
    )
    keywords_response.raise_for_status()

    result["keywords"] = [
        keyword["name"]
        for keyword in keywords_response.json().get("keywords", [])
    ]

    return result

# Convert genre IDs into readable names
movies_df["genre_ids"] = movies_df["genre_ids"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

movies_df["genres"] = movies_df["genre_ids"].apply(
    lambda ids: [
        genre_map[genre_id]
        for genre_id in (ids if isinstance(ids, list) else [])
        if genre_id in genre_map
    ]
)

# Start with a small batch to verify the process
# Increase to len(movies_df) only after the test succeeds
TEST_LIMIT = min(600, len(movies_df))

metadata = []
failed_ids = []

for i, row in movies_df.head(TEST_LIMIT).iterrows():
    try:
        details = get_movie_metadata(row["tmdb_id"])
        metadata.append({
            "tmdb_id": int(row["tmdb_id"]),
            **details
        })
        print(f"Processed {i + 1}/{TEST_LIMIT}: {row['title']}")

    except (requests.RequestException, ValueError, KeyError) as error:
        failed_ids.append(row["tmdb_id"])
        metadata.append({
            "tmdb_id": int(row["tmdb_id"]),
            "cast": [],
            "director": None,
            "keywords": []
        })
        print(f"Failed: {row['title']} - {error}")

    time.sleep(0.15)

metadata_df = pd.DataFrame(metadata)

# Merge without losing raw fields such as poster_path
movies_enriched = movies_df.merge(
    metadata_df,
    on="tmdb_id",
    how="left",
    validate="one_to_one"
)

print("\nTest enrichment complete.")
print("Enriched rows:", len(movies_enriched))
print("Failed movie IDs:", failed_ids)
display(
    movies_enriched[
        ["tmdb_id", "title", "genres", "cast",
         "director", "keywords", "poster_path"]
    ].head()
)


Processed 1/600: Spider-Man: Brand New Day
Processed 2/600: Resident Evil
Processed 3/600: Insidious: Out of the Further
Processed 4/600: The Odyssey
Processed 5/600: The Fix
Processed 6/600: Runner
Processed 7/600: Fall 2: Deadpoint
Processed 8/600: After Impact
Processed 9/600: Animals
Processed 10/600: Resident Dead
Processed 11/600: Resident Evil: Welcome to Raccoon City
Processed 12/600: Coyote vs. Acme
Processed 13/600: The Mongoose
Processed 14/600: The End of Oak Street
Processed 15/600: Matchbox the Movie
Processed 16/600: Toy Story 5
Processed 17/600: Digger
Processed 18/600: Pinocchio: Unstrung
Processed 19/600: Obsession
Processed 20/600: The Love Hypothesis
Processed 21/600: Mutiny
Processed 22/600: Onslaught
Processed 23/600: The Uprising
Processed 24/600: Other Mommy
Processed 25/600: Before I Go to Sleep
Processed 26/600: Zip Wire
Processed 27/600: Verity
Processed 28/600: Moana
Processed 29/600: Heart of the Beast
Processed 30/600: Blaze of Love
Processed 31/600: Zooto

,tmdb_id,title,genres,cast,director,keywords,poster_path
0,969681,Spider-Man: Brand New Day,"[Science Fiction, Action, Adventure]","[Tom Holland, Sadie Sink, Tramell Tillman, Zen...",Destin Daniel Cretton,"[mind control, new york city, hero, mutation, ...",/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg
1,1423191,Resident Evil,"[Horror, Science Fiction, Adventure]","[Austin Abrams, Paul Walter Hauser, Kali Reis,...",Zach Cregger,"[outbreak, survival, decapitation, zombie, bas...",/i7UyjfPio0VFHB9rBUZSFyhOoM8.jpg
2,1291595,Insidious: Out of the Further,"[Horror, Thriller]","[Amelia Eve, Island Austin, Brandon Perea, Lin...",Jacob Chase,"[supernatural, trauma, cult, sequel, evil spir...",/4tTrW9dXCByS5wt2pXVWb58zNjz.jpg
3,1368337,The Odyssey,"[Adventure, Action, Fantasy]","[Matt Damon, Tom Holland, Anne Hathaway, Rober...",Christopher Nolan,"[ship, trojan war, greek mythology, historical...",/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg
4,1228834,The Fix,"[Action, Thriller]","[Zachary Levi, Liam Neeson, Wes Chatham, Titus...",Guy Moshe,"[central intelligence agency (cia), war vetera...",/yopXjun3ICFfJci2ukcEzceZjUs.jpg


In [ ]:
print("movies_enriched exists:", "movies_enriched" in globals())

if "movies_enriched" in globals():
    print("Shape:", movies_enriched.shape)
    print("Columns:", movies_enriched.columns.tolist())

    display(movies_enriched[
        ["tmdb_id", "title", "genres", "cast",
         "director", "keywords", "poster_path"]
    ].head())

movies_enriched exists: True
Shape: (600, 18)
Columns: ['tmdb_id', 'title', 'original_title', 'original_language', 'release_date', 'overview', 'genre_ids', 'vote_average', 'vote_count', 'popularity', 'poster_path', 'adult', 'original_language_name', 'poster_url', 'genres', 'cast', 'director', 'keywords']


,tmdb_id,title,genres,cast,director,keywords,poster_path
0,969681,Spider-Man: Brand New Day,"[Science Fiction, Action, Adventure]","[Tom Holland, Sadie Sink, Tramell Tillman, Zen...",Destin Daniel Cretton,"[mind control, new york city, hero, mutation, ...",/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg
1,1423191,Resident Evil,"[Horror, Science Fiction, Adventure]","[Austin Abrams, Paul Walter Hauser, Kali Reis,...",Zach Cregger,"[outbreak, survival, decapitation, zombie, bas...",/i7UyjfPio0VFHB9rBUZSFyhOoM8.jpg
2,1291595,Insidious: Out of the Further,"[Horror, Thriller]","[Amelia Eve, Island Austin, Brandon Perea, Lin...",Jacob Chase,"[supernatural, trauma, cult, sequel, evil spir...",/4tTrW9dXCByS5wt2pXVWb58zNjz.jpg
3,1368337,The Odyssey,"[Adventure, Action, Fantasy]","[Matt Damon, Tom Holland, Anne Hathaway, Rober...",Christopher Nolan,"[ship, trojan war, greek mythology, historical...",/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg
4,1228834,The Fix,"[Action, Thriller]","[Zachary Levi, Liam Neeson, Wes Chatham, Titus...",Guy Moshe,"[central intelligence agency (cia), war vetera...",/yopXjun3ICFfJci2ukcEzceZjUs.jpg


In [ ]:
#saved the enrich dataset
import json

movies_enriched.to_json(
    "/content/movies_enriched.json",
    orient="records",
    force_ascii=False,
    indent=2
)

movies_enriched.to_csv(
    "/content/movies_enriched.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Enriched JSON and CSV saved successfully.")

Enriched JSON and CSV saved successfully.


In [ ]:
# Check whether poster information exists
print("Poster column exists:", "poster_path" in movies_df.columns)

if "poster_path" in movies_df.columns:
    missing = (
        movies_df["poster_path"].isna()
        | movies_df["poster_path"].astype(str).str.strip().isin(["", "None", "nan"])
    )

    print("Total movies:", len(movies_df))
    print("Movies with poster paths:", int((~missing).sum()))
    print("Movies missing posters:", int(missing.sum()))

    display(
        movies_df[
            ["tmdb_id", "title", "poster_path"]
        ].head(10)
    )
else:
    print("poster_path is missing. We need to restore it from the raw TMDB data.")

Poster column exists: True
Total movies: 600
Movies with poster paths: 547
Movies missing posters: 53


,tmdb_id,title,poster_path
0,969681,Spider-Man: Brand New Day,/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg
1,1423191,Resident Evil,/i7UyjfPio0VFHB9rBUZSFyhOoM8.jpg
2,1291595,Insidious: Out of the Further,/4tTrW9dXCByS5wt2pXVWb58zNjz.jpg
3,1368337,The Odyssey,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg
4,1228834,The Fix,/yopXjun3ICFfJci2ukcEzceZjUs.jpg
5,1377237,Runner,/yBKMAIj7clP42UkFejhGDBBoTpb.jpg
6,1101412,Fall 2: Deadpoint,/fgSm5ylwiXbIHn8UbUXDjk9RRu4.jpg
7,1751701,After Impact,/yzDuAQCY2n6zzOlBkn6JzhLhY3k.jpg
8,1236045,Animals,/cyltXU1lAGhRFBuHQIMWBAPj8fI.jpg
9,1712451,Resident Dead,/qKhMVWwFikSUzOpkhOl35GBLkGH.jpg


In [ ]:
#making the poster urlas seperate feild
import pandas as pd

def make_poster_url(path):
    if pd.isna(path) or not str(path).strip():
        return None

    path = str(path).strip()

    if not path.startswith("/"):
        path = "/" + path

    return f"https://image.tmdb.org/t/p/w500{path}"

movies_df["poster_url"] = movies_df["poster_path"].apply(
    make_poster_url
)

display(
    movies_df[
        ["tmdb_id", "title", "poster_path", "poster_url"]
    ].head(10)
)

,tmdb_id,title,poster_path,poster_url
0,969681,Spider-Man: Brand New Day,/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg,https://image.tmdb.org/t/p/w500/bjiS5ipwxb9JFy...
1,1423191,Resident Evil,/i7UyjfPio0VFHB9rBUZSFyhOoM8.jpg,https://image.tmdb.org/t/p/w500/i7UyjfPio0VFHB...
2,1291595,Insidious: Out of the Further,/4tTrW9dXCByS5wt2pXVWb58zNjz.jpg,https://image.tmdb.org/t/p/w500/4tTrW9dXCByS5w...
3,1368337,The Odyssey,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,https://image.tmdb.org/t/p/w500/5rhTDKUhPYvpdQ...
4,1228834,The Fix,/yopXjun3ICFfJci2ukcEzceZjUs.jpg,https://image.tmdb.org/t/p/w500/yopXjun3ICFfJc...
5,1377237,Runner,/yBKMAIj7clP42UkFejhGDBBoTpb.jpg,https://image.tmdb.org/t/p/w500/yBKMAIj7clP42U...
6,1101412,Fall 2: Deadpoint,/fgSm5ylwiXbIHn8UbUXDjk9RRu4.jpg,https://image.tmdb.org/t/p/w500/fgSm5ylwiXbIHn...
7,1751701,After Impact,/yzDuAQCY2n6zzOlBkn6JzhLhY3k.jpg,https://image.tmdb.org/t/p/w500/yzDuAQCY2n6zzO...
8,1236045,Animals,/cyltXU1lAGhRFBuHQIMWBAPj8fI.jpg,https://image.tmdb.org/t/p/w500/cyltXU1lAGhRFB...
9,1712451,Resident Dead,/qKhMVWwFikSUzOpkhOl35GBLkGH.jpg,https://image.tmdb.org/t/p/w500/qKhMVWwFikSUzO...


In [ ]:
#ensuring it works or not
from IPython.display import display, Image

movie = movies_df[
    movies_df["poster_url"].notna()
].iloc[0]

print(movie["title"])
display(Image(url=movie["poster_url"], width=200))

Spider-Man: Brand New Day


In [ ]:
#feature engineering
#verify the enrich dataset
import pandas as pd
import json
import ast

movies_df = pd.read_csv("/content/movies_enriched.csv")

print("Dataset shape:", movies_df.shape)
print("Columns:", movies_df.columns.tolist())
display(movies_df.head(3))

Dataset shape: (600, 18)
Columns: ['tmdb_id', 'title', 'original_title', 'original_language', 'release_date', 'overview', 'genre_ids', 'vote_average', 'vote_count', 'popularity', 'poster_path', 'adult', 'original_language_name', 'poster_url', 'genres', 'cast', 'director', 'keywords']


,tmdb_id,title,original_title,original_language,release_date,overview,genre_ids,vote_average,vote_count,popularity,poster_path,adult,original_language_name,poster_url,genres,cast,director,keywords
0,969681,Spider-Man: Brand New Day,Spider-Man: Brand New Day,en,2026-07-29,Fighting crime full-time as Spider-Man in a wo...,"[878, 28, 12]",8.4,4936,986.0191,/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg,False,English,https://image.tmdb.org/t/p/w500/bjiS5ipwxb9JFy...,"['Science Fiction', 'Action', 'Adventure']","['Tom Holland', 'Sadie Sink', 'Tramell Tillman...",Destin Daniel Cretton,"['mind control', 'new york city', 'hero', 'mut..."
1,1423191,Resident Evil,Resident Evil,en,2026-09-16,Medical courier Bryan unwittingly finds himsel...,"[27, 878, 12]",7.3,859,704.1066,/i7UyjfPio0VFHB9rBUZSFyhOoM8.jpg,False,English,https://image.tmdb.org/t/p/w500/i7UyjfPio0VFHB...,"['Horror', 'Science Fiction', 'Adventure']","['Austin Abrams', 'Paul Walter Hauser', 'Kali ...",Zach Cregger,"['outbreak', 'survival', 'decapitation', 'zomb..."
2,1291595,Insidious: Out of the Further,Insidious: Out of the Further,en,2026-08-19,"Gemma, a young mother raising her daughter in ...","[27, 53]",7.6,568,454.8876,/4tTrW9dXCByS5wt2pXVWb58zNjz.jpg,False,English,https://image.tmdb.org/t/p/w500/4tTrW9dXCByS5w...,"['Horror', 'Thriller']","['Amelia Eve', 'Island Austin', 'Brandon Perea...",Jacob Chase,"['supernatural', 'trauma', 'cult', 'sequel', '..."


In [ ]:
#Convert metadata into usable text
def parse_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value) or str(value).strip() == "":
        return []

    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, TypeError):
        pass

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass

    return []

for col in ["genres", "cast", "keywords"]:
    movies_df[col] = movies_df[col].apply(parse_list)

print("Metadata converted successfully.")

Metadata converted successfully.


In [ ]:
#create tags on columns
def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()

def build_tags(row):
    parts = []

    parts.extend(row["genres"])
    parts.extend(row["keywords"])
    parts.extend(row["cast"][:5])

    if pd.notna(row["director"]):
        parts.append(str(row["director"]))

    overview = clean_text(row["overview"])
    if overview:
        parts.append(overview)

    return " ".join(parts).lower()

movies_df["tags"] = movies_df.apply(build_tags, axis=1)

display(
    movies_df[["title", "genres", "director", "tags"]].head(10)
)

,title,genres,director,tags
0,Spider-Man: Brand New Day,"[Science Fiction, Action, Adventure]",Destin Daniel Cretton,science fiction action adventure mind control ...
1,Resident Evil,"[Horror, Science Fiction, Adventure]",Zach Cregger,horror science fiction adventure outbreak surv...
2,Insidious: Out of the Further,"[Horror, Thriller]",Jacob Chase,horror thriller supernatural trauma cult seque...
3,The Odyssey,"[Adventure, Action, Fantasy]",Christopher Nolan,adventure action fantasy ship trojan war greek...
4,The Fix,"[Action, Thriller]",Guy Moshe,action thriller central intelligence agency (c...
5,Runner,"[Action, Comedy]",Scott Waugh,action comedy race against time black market d...
6,Fall 2: Deadpoint,[Thriller],Michael Spierig,thriller sequel rock climbing sport climbing c...
7,After Impact,"[Science Fiction, Drama, Thriller]",Michiel Rensen,science fiction drama thriller asteroid loss l...
8,Animals,"[Thriller, Drama]",Ben Affleck,thriller drama kidnapping political candidate ...
9,Resident Dead,[Horror],Ryan Ebert,horror mockbuster enchant john wells isabel ra...


In [ ]:
#Save the feature-engineered dataset
movies_df.to_csv(
    "/content/movies_featured.csv",
    index=False,
    encoding="utf-8-sig"
)

movies_df.to_json(
    "/content/movies_featured.json",
    orient="records",
    force_ascii=False,
    indent=2
)

print("Feature engineering completed.")
print("Saved movies_featured.csv and movies_featured.json")

Feature engineering completed.
Saved movies_featured.csv and movies_featured.json


In [ ]:
#validate before tf-idf
print("Total movies:", len(movies_df))
print("Missing titles:", movies_df["title"].isna().sum())
print("Empty tags:", (movies_df["tags"].str.strip() == "").sum())
print("Duplicate IDs:", movies_df["tmdb_id"].duplicated().sum())

display(movies_df[["title", "tags"]].head(5))

Total movies: 600
Missing titles: 0
Empty tags: 0
Duplicate IDs: 0


,title,tags
0,Spider-Man: Brand New Day,science fiction action adventure mind control ...
1,Resident Evil,horror science fiction adventure outbreak surv...
2,Insidious: Out of the Further,horror thriller supernatural trauma cult seque...
3,The Odyssey,adventure action fantasy ship trojan war greek...
4,The Fix,action thriller central intelligence agency (c...


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
movies_df = pd.read_csv("/content/movies_featured.csv")

print("Dataset shape:", movies_df.shape)
print("Available columns:", movies_df.columns.tolist())

movies_df["tags"] = movies_df["tags"].fillna("").astype(str)

display(movies_df[["title", "tags"]].head())

Dataset shape: (600, 19)
Available columns: ['tmdb_id', 'title', 'original_title', 'original_language', 'release_date', 'overview', 'genre_ids', 'vote_average', 'vote_count', 'popularity', 'poster_path', 'adult', 'original_language_name', 'poster_url', 'genres', 'cast', 'director', 'keywords', 'tags']


,title,tags
0,Spider-Man: Brand New Day,science fiction action adventure mind control ...
1,Resident Evil,horror science fiction adventure outbreak surv...
2,Insidious: Out of the Further,horror thriller supernatural trauma cult seque...
3,The Odyssey,adventure action fantasy ship trojan war greek...
4,The Fix,action thriller central intelligence agency (c...


In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = tfidf.fit_transform(movies_df["tags"])

print("TF-IDF completed successfully.")
print("Number of movies:", tfidf_matrix.shape[0])
print("Number of features:", tfidf_matrix.shape[1])
print("Matrix type:", type(tfidf_matrix))

TF-IDF completed successfully.
Number of movies: 600
Number of features: 10000
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>


In [ ]:
feature_names = tfidf.get_feature_names_out()

print("First 30 TF-IDF features:")
print(feature_names[:30])

print("\nFirst movie:", movies_df.iloc[0]["title"])
print("Non-zero features:", tfidf_matrix[0].nnz)

First 30 TF-IDF features:
['000 prize' '10' '10 100' '10 day' '100' '100 000' '100 daily' '11'
 '11 rizwan' '11 year' '11th' '11th birthday' '12' '12 year' '12 years'
 '1392' '1392 1910' '14' '14 year' '14th' '14th century' '15' '15 year'
 '15 years' '17th' '17th century' '18' '1910' '1920s' '1926']

First movie: Spider-Man: Brand New Day
Non-zero features: 96


In [ ]:
import joblib

joblib.dump(tfidf, "/content/tfidf_vectorizer.pkl")

print("Saved tfidf_vectorizer.pkl")

Saved tfidf_vectorizer.pkl


In [ ]:
#calculate cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between every pair of movies
cosine_sim = cosine_similarity(tfidf_matrix, dense_output=False)

print("Cosine similarity calculated successfully.")
print("Similarity matrix shape:", cosine_sim.shape)
print("Number of movies:", len(movies_df))

Cosine similarity calculated successfully.
Similarity matrix shape: (600, 600)
Number of movies: 600


In [ ]:
# Inspect similarity scores for the first movie
movie_index = 0

scores = cosine_sim[movie_index].toarray().flatten()

similar_movies = sorted(
    enumerate(scores),
    key=lambda x: x[1],
    reverse=True
)

for index, score in similar_movies[:10]:
    print(f"{movies_df.iloc[index]['title']}: {score:.4f}")

Spider-Man: Brand New Day: 1.0000
Spider-Man: No Way Home: 0.5202
The Avengers: 0.1945
Avengers: Doomsday: 0.1838
Avengers: Infinity War: 0.1748
Avengers: Endgame: 0.1517
Deadpool & Wolverine: 0.1422
My Hero Academia: You're Next: 0.1075
Minnal Murali: 0.0847
Disclosure Day: 0.0833


In [ ]:
from scipy.sparse import save_npz

save_npz("/content/cosine_similarity.npz", cosine_sim)

print("Saved cosine_similarity.npz")

Saved cosine_similarity.npz


In [ ]:
#build content based recommender
def recommend_movies(movie_title, top_n=10):
    matches = movies_df.index[
        movies_df["title"].astype(str).str.casefold()
        == movie_title.strip().casefold()
    ].tolist()

    if not matches:
        print(f"Movie not found: {movie_title}")
        return pd.DataFrame()

    # If duplicate titles exist, use the first matching record
    movie_index = matches[0]

    scores = cosine_sim[movie_index].toarray().flatten()

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )

    # Exclude the selected movie itself
    ranked_indices = [
        i for i in ranked_indices
        if i != movie_index
    ][:top_n]

    result = movies_df.iloc[ranked_indices].copy()
    result["similarity_score"] = [
        float(scores[i]) for i in ranked_indices
    ]

    columns = [
        col for col in [
            "tmdb_id", "title", "genres", "overview",
            "vote_average", "poster_path", "poster_url",
            "similarity_score"
        ]
        if col in result.columns
    ]

    return result[columns].reset_index(drop=True)

In [ ]:
#test first recommendation
recommendations = recommend_movies(
    movies_df.iloc[0]["title"],
    top_n=10
)

display(recommendations)

,tmdb_id,title,genres,overview,vote_average,poster_path,poster_url,similarity_score
0,634649,Spider-Man: No Way Home,"['Action', 'Adventure', 'Science Fiction']",Peter Parker is unmasked and no longer able to...,7.947,/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg,https://image.tmdb.org/t/p/w500/1g0dhYtq4irTY1...,0.520230
1,24428,The Avengers,"['Science Fiction', 'Action', 'Adventure']",When an unexpected enemy emerges and threatens...,8.080,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,https://image.tmdb.org/t/p/w500/RYMX2wcKCBAr24...,0.194500
2,1003596,Avengers: Doomsday,"['Science Fiction', 'Action', 'Adventure']",Beloved heroes from three distinct universes a...,0.000,/jzPwsojjFStf5lR5Nm07w2hH56G.jpg,https://image.tmdb.org/t/p/w500/jzPwsojjFStf5l...,0.183803
3,299536,Avengers: Infinity War,"['Adventure', 'Action', 'Science Fiction']",As the Avengers and their allies have continue...,8.244,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,https://image.tmdb.org/t/p/w500/7WsyChQLEftFiD...,0.174790
4,299534,Avengers: Endgame,"['Adventure', 'Science Fiction', 'Action']",After the devastating events of Avengers: Infi...,8.245,/ulzhLuWrPK07P1YkdWQLZnQh1JL.jpg,https://image.tmdb.org/t/p/w500/ulzhLuWrPK07P1...,0.151692
5,533535,Deadpool & Wolverine,"['Action', 'Comedy', 'Science Fiction']",A listless Wade Wilson toils away in civilian ...,7.585,/8cdWjvZQUExUUTzyp4t6EDMubfO.jpg,https://image.tmdb.org/t/p/w500/8cdWjvZQUExUUT...,0.142216
6,1159311,My Hero Academia: You're Next,"['Animation', 'Action', 'Adventure', 'Science ...",In a society devastated by the effects of an a...,7.300,/tTrI6PwqzxkgO3dvQ7BEKXM7SYR.jpg,https://image.tmdb.org/t/p/w500/tTrI6PwqzxkgO3...,0.107525
7,657644,Minnal Murali,"['Action', 'Comedy', 'Fantasy']",A tailor gains special powers after being stru...,7.200,/efetKFDyptrRpoHBb103Tg3Auw5.jpg,https://image.tmdb.org/t/p/w500/efetKFDyptrRpo...,0.084746
8,1275779,Disclosure Day,"['Science Fiction', 'Thriller']",A cybersecurity expert becomes a whistleblower...,7.400,/AnJ8IQJI23hNpYXVNaythu061Ru.jpg,https://image.tmdb.org/t/p/w500/AnJ8IQJI23hNpY...,0.083266
9,149,Akira,"['Animation', 'Science Fiction', 'Action']",A secret military project endangers Neo-Tokyo ...,7.938,/neZ0ykEsPqxamsX6o5QNUFILQrz.jpg,https://image.tmdb.org/t/p/w500/neZ0ykEsPqxams...,0.082073


In [ ]:
recommendations = recommend_movies("Spider-Man: Brand New Day")
display(recommendations)

,tmdb_id,title,genres,overview,vote_average,poster_path,poster_url,similarity_score
0,634649,Spider-Man: No Way Home,"['Action', 'Adventure', 'Science Fiction']",Peter Parker is unmasked and no longer able to...,7.947,/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg,https://image.tmdb.org/t/p/w500/1g0dhYtq4irTY1...,0.520230
1,24428,The Avengers,"['Science Fiction', 'Action', 'Adventure']",When an unexpected enemy emerges and threatens...,8.080,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,https://image.tmdb.org/t/p/w500/RYMX2wcKCBAr24...,0.194500
2,1003596,Avengers: Doomsday,"['Science Fiction', 'Action', 'Adventure']",Beloved heroes from three distinct universes a...,0.000,/jzPwsojjFStf5lR5Nm07w2hH56G.jpg,https://image.tmdb.org/t/p/w500/jzPwsojjFStf5l...,0.183803
3,299536,Avengers: Infinity War,"['Adventure', 'Action', 'Science Fiction']",As the Avengers and their allies have continue...,8.244,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,https://image.tmdb.org/t/p/w500/7WsyChQLEftFiD...,0.174790
4,299534,Avengers: Endgame,"['Adventure', 'Science Fiction', 'Action']",After the devastating events of Avengers: Infi...,8.245,/ulzhLuWrPK07P1YkdWQLZnQh1JL.jpg,https://image.tmdb.org/t/p/w500/ulzhLuWrPK07P1...,0.151692
5,533535,Deadpool & Wolverine,"['Action', 'Comedy', 'Science Fiction']",A listless Wade Wilson toils away in civilian ...,7.585,/8cdWjvZQUExUUTzyp4t6EDMubfO.jpg,https://image.tmdb.org/t/p/w500/8cdWjvZQUExUUT...,0.142216
6,1159311,My Hero Academia: You're Next,"['Animation', 'Action', 'Adventure', 'Science ...",In a society devastated by the effects of an a...,7.300,/tTrI6PwqzxkgO3dvQ7BEKXM7SYR.jpg,https://image.tmdb.org/t/p/w500/tTrI6PwqzxkgO3...,0.107525
7,657644,Minnal Murali,"['Action', 'Comedy', 'Fantasy']",A tailor gains special powers after being stru...,7.200,/efetKFDyptrRpoHBb103Tg3Auw5.jpg,https://image.tmdb.org/t/p/w500/efetKFDyptrRpo...,0.084746
8,1275779,Disclosure Day,"['Science Fiction', 'Thriller']",A cybersecurity expert becomes a whistleblower...,7.400,/AnJ8IQJI23hNpYXVNaythu061Ru.jpg,https://image.tmdb.org/t/p/w500/AnJ8IQJI23hNpY...,0.083266
9,149,Akira,"['Animation', 'Science Fiction', 'Action']",A secret military project endangers Neo-Tokyo ...,7.938,/neZ0ykEsPqxamsX6o5QNUFILQrz.jpg,https://image.tmdb.org/t/p/w500/neZ0ykEsPqxams...,0.082073


In [ ]:
#Create a user interaction history
import pandas as pd

user_history = pd.DataFrame([
    {"title": "Spider-Man: Brand New Day", "action": "like"},
    {"title": "Avengers: Endgame", "action": "like"},
    {"title": "The Notebook", "action": "skip"}
])

display(user_history)

,title,action
0,Spider-Man: Brand New Day,like
1,Avengers: Endgame,like
2,The Notebook,skip


In [ ]:
#Build the user's genre preference profile
ACTION_WEIGHTS = {
    "like": 1.0,
    "dislike": -1.0,
    "skip": -0.25
}

def build_user_profile(user_history, movies_df):
    genre_scores = {}

    for _, interaction in user_history.iterrows():
        title = interaction["title"]
        action = interaction["action"].lower()

        if action not in ACTION_WEIGHTS:
            continue

        matches = movies_df[
            movies_df["title"].str.casefold() == title.casefold()
        ]

        if matches.empty:
            print(f"Movie not found in dataset: {title}")
            continue

        movie = matches.iloc[0]
        genres = movie["genres"]

        if isinstance(genres, str):
            import ast
            try:
                genres = ast.literal_eval(genres)
            except (ValueError, SyntaxError):
                genres = [genres]

        weight = ACTION_WEIGHTS[action]

        for genre in genres:
            genre_scores[genre] = (
                genre_scores.get(genre, 0) + weight
            )

    return genre_scores

user_profile = build_user_profile(user_history, movies_df)

print("User genre preference scores:")
print(user_profile)

Movie not found in dataset: The Notebook
User genre preference scores:
{'Science Fiction': 2.0, 'Action': 2.0, 'Adventure': 2.0}


In [ ]:
#Calculate a preference score for each movie
def calculate_preference_score(movie, user_profile):
    genres = movie["genres"]

    if isinstance(genres, str):
        import ast
        try:
            genres = ast.literal_eval(genres)
        except (ValueError, SyntaxError):
            genres = [genres]

    if not genres:
        return 0.0

    scores = [user_profile.get(genre, 0.0) for genre in genres]

    # Normalize the score to approximately [-1, 1]
    return sum(scores) / len(scores)

movies_df["preference_score"] = movies_df.apply(
    lambda movie: calculate_preference_score(movie, user_profile),
    axis=1
)

display(
    movies_df[
        ["title", "genres", "preference_score"]
    ].sort_values("preference_score", ascending=False).head(10)
)

,title,genres,preference_score
0,Spider-Man: Brand New Day,"['Science Fiction', 'Action', 'Adventure']",2.0
36,Avengers: Infinity War,"['Adventure', 'Action', 'Science Fiction']",2.0
32,Avengers: Endgame,"['Adventure', 'Science Fiction', 'Action']",2.0
31,Project Hail Mary,"['Science Fiction', 'Adventure']",2.0
33,Avengers: Doomsday,"['Science Fiction', 'Action', 'Adventure']",2.0
44,Lion Fist,['Action'],2.0
50,The Avengers,"['Science Fiction', 'Action', 'Adventure']",2.0
457,45,['Action'],2.0
56,Dune: Part Two,"['Science Fiction', 'Adventure']",2.0
55,The Mandalorian and Grogu,"['Action', 'Adventure', 'Science Fiction']",2.0


In [ ]:
#Add mood and context scoring in Colab
import pandas as pd
import ast
import json

# Mood-to-genre mapping
MOOD_GENRES = {
    "happy": ["Comedy", "Animation", "Family"],
    "romantic": ["Romance", "Comedy", "Drama"],
    "thrilling": ["Thriller", "Action", "Crime"],
    "emotional": ["Drama", "Romance"],
    "thought-provoking": [
        "Science Fiction", "Drama", "Mystery"
    ],
    "funny": ["Comedy", "Animation"]
}

# Context-to-genre mapping
CONTEXT_GENRES = {
    "solo": ["Drama", "Mystery", "Thriller"],
    "family": ["Family", "Animation", "Comedy"],
    "weekend": ["Adventure", "Comedy", "Action"],
    "late-night": ["Mystery", "Thriller", "Drama"]
}

def parse_genres(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, TypeError):
        pass

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

movies_df["genres"] = movies_df["genres"].apply(parse_genres)

def genre_match_score(movie_genres, target_genres):
    if not movie_genres or not target_genres:
        return 0.0

    matches = set(movie_genres) & set(target_genres)
    return len(matches) / len(set(target_genres))

def calculate_mood_score(genres, mood):
    target = MOOD_GENRES.get(mood.lower(), [])
    return genre_match_score(genres, target)

def calculate_context_score(movie, context):
    context = context.lower()
    genres = movie["genres"]

    if context in CONTEXT_GENRES:
        return genre_match_score(
            genres, CONTEXT_GENRES[context]
        )

    if context == "quick watch":
        # Runtime must come from TMDB movie details.
        runtime = movie.get("runtime", None)

        if pd.isna(runtime) or runtime is None:
            return 0.0

        return 1.0 if runtime <= 100 else 0.0

    if context == "rewatch":
        liked_titles = set(
            user_history.loc[
                user_history["action"].str.lower() == "like",
                "title"
            ].str.casefold()
        )

        return (
            1.0
            if str(movie["title"]).casefold() in liked_titles
            else 0.0
        )

    return 0.0

print("Mood and context scoring functions created.")
print("Available moods:", list(MOOD_GENRES.keys()))
print(
    "Available contexts:",
    list(CONTEXT_GENRES.keys()) +
    ["quick watch", "rewatch"]
)

Mood and context scoring functions created.
Available moods: ['happy', 'romantic', 'thrilling', 'emotional', 'thought-provoking', 'funny']
Available contexts: ['solo', 'family', 'weekend', 'late-night', 'quick watch', 'rewatch']


In [ ]:
#le user select the mood
selected_mood = "happy"
selected_context = "weekend"

movies_df["mood_score"] = movies_df["genres"].apply(
    lambda genres: calculate_mood_score(
        genres, selected_mood
    )
)

movies_df["context_score"] = movies_df.apply(
    lambda movie: calculate_context_score(
        movie, selected_context
    ),
    axis=1
)

print("Selected mood:", selected_mood)
print("Selected context:", selected_context)

display(
    movies_df[
        ["title", "genres", "mood_score", "context_score"]
    ].sort_values(
        ["mood_score", "context_score"],
        ascending=False
    ).head(10)
)

Selected mood: happy
Selected context: weekend


,title,genres,mood_score,context_score
15,Toy Story 5,"[Animation, Family, Comedy, Adventure]",1.000000,0.666667
30,Zootopia 2,"[Animation, Adventure, Comedy, Mystery, Family]",1.000000,0.666667
35,Minions & Monsters,"[Adventure, Animation, Comedy, Family, Fantasy]",1.000000,0.666667
167,Red Shoes and the Seven Dwarfs,"[Animation, Fantasy, Family, Comedy, Romance, ...",1.000000,0.666667
11,Coyote vs. Acme,"[Comedy, Adventure, Family]",0.666667,0.666667
27,Moana,"[Family, Fantasy, Comedy, Adventure]",0.666667,0.666667
245,Aadarsha Kutumbam House No: 47 - AK47,"[Family, Action, Comedy]",0.666667,0.666667
554,Crayon Shin-chan the Movie: Very Bizarre! My Y...,"[Adventure, Animation, Comedy]",0.666667,0.666667
569,Castle in the Sky,"[Adventure, Fantasy, Animation, Action, Family]",0.666667,0.666667
85,Dangal,"[Drama, Family, Comedy]",0.666667,0.333333


In [ ]:
import numpy as np

# Build a title-to-index lookup
title_to_indices = {}

for idx, title in enumerate(movies_df["title"].astype(str)):
    title_to_indices.setdefault(title.casefold(), []).append(idx)

liked_titles = user_history.loc[
    user_history["action"].str.lower() == "like",
    "title"
].astype(str).str.casefold().tolist()

liked_indices = [
    idx
    for title in liked_titles
    for idx in title_to_indices.get(title, [])
]

if not liked_indices:
    raise ValueError(
        "No liked movies were found in the dataset. "
        "Add valid movie titles to user_history first."
    )

# Content similarity to movies the user liked
liked_similarity = cosine_sim[:, liked_indices].toarray()

movies_df["content_score"] = liked_similarity.max(axis=1)

# Normalize user preference scores to 0–1
pref_min = movies_df["preference_score"].min()
pref_max = movies_df["preference_score"].max()

if pref_max > pref_min:
    movies_df["preference_normalized"] = (
        (movies_df["preference_score"] - pref_min)
        / (pref_max - pref_min)
    )
else:
    movies_df["preference_normalized"] = 0.5

# Normalize ratings to 0–1 using TMDB's 0–10 rating scale
movies_df["rating_normalized"] = (
    pd.to_numeric(movies_df["vote_average"], errors="coerce")
    .fillna(0)
    .clip(0, 10) / 10
)

# Initial weighted final score
movies_df["final_score"] = (
    0.40 * movies_df["content_score"]
    + 0.25 * movies_df["preference_normalized"]
    + 0.15 * movies_df["mood_score"]
    + 0.10 * movies_df["context_score"]
    + 0.10 * movies_df["rating_normalized"]
)

# Exclude movies the user has already liked from new suggestions
liked_set = set(liked_titles)

recommendations = movies_df[
    ~movies_df["title"].astype(str).str.casefold().isin(liked_set)
].sort_values("final_score", ascending=False)

columns_to_show = [
    col for col in [
        "tmdb_id", "title", "genres", "vote_average",
        "poster_path", "poster_url", "content_score",
        "preference_normalized", "mood_score",
        "context_score", "rating_normalized", "final_score"
    ]
    if col in recommendations.columns
]

display(recommendations[columns_to_show].head(10))

,tmdb_id,title,genres,vote_average,poster_path,poster_url,content_score,preference_normalized,mood_score,context_score,rating_normalized,final_score
49,634649,Spider-Man: No Way Home,"[Action, Adventure, Science Fiction]",7.947,/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg,https://image.tmdb.org/t/p/w500/1g0dhYtq4irTY1...,0.520230,1.000000,0.000000,0.666667,0.7947,0.604228
50,24428,The Avengers,"[Science Fiction, Action, Adventure]",8.080,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,https://image.tmdb.org/t/p/w500/RYMX2wcKCBAr24...,0.453151,1.000000,0.000000,0.666667,0.8080,0.578727
36,299536,Avengers: Infinity War,"[Adventure, Action, Science Fiction]",8.244,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,https://image.tmdb.org/t/p/w500/7WsyChQLEftFiD...,0.416990,1.000000,0.000000,0.666667,0.8244,0.565903
33,1003596,Avengers: Doomsday,"[Science Fiction, Action, Adventure]",0.000,/jzPwsojjFStf5lR5Nm07w2hH56G.jpg,https://image.tmdb.org/t/p/w500/jzPwsojjFStf5l...,0.428079,1.000000,0.000000,0.666667,0.0000,0.487898
588,1159311,My Hero Academia: You're Next,"[Animation, Action, Adventure, Science Fiction]",7.300,/tTrI6PwqzxkgO3dvQ7BEKXM7SYR.jpg,https://image.tmdb.org/t/p/w500/tTrI6PwqzxkgO3...,0.107525,0.750000,0.333333,0.666667,0.7300,0.420176
55,1228710,The Mandalorian and Grogu,"[Action, Adventure, Science Fiction]",7.500,/uwMKWjcNID0D9jjplsjkQS2OrB4.jpg,https://image.tmdb.org/t/p/w500/uwMKWjcNID0D9j...,0.062401,1.000000,0.000000,0.666667,0.7500,0.416627
59,533535,Deadpool & Wolverine,"[Action, Comedy, Science Fiction]",7.585,/8cdWjvZQUExUUTzyp4t6EDMubfO.jpg,https://image.tmdb.org/t/p/w500/8cdWjvZQUExUUT...,0.142216,0.666667,0.333333,0.666667,0.7585,0.416070
14,1148365,Matchbox the Movie,"[Action, Comedy, Adventure]",7.857,/lFKA4GMz36dImEy3DQPdPF7G4V9.jpg,https://image.tmdb.org/t/p/w500/lFKA4GMz36dImE...,0.038222,0.666667,0.333333,1.000000,0.7857,0.410525
31,687163,Project Hail Mary,"[Science Fiction, Adventure]",8.633,/iOb2fjXLbpJgyQXe46n1WtGCnaa.jpg,https://image.tmdb.org/t/p/w500/iOb2fjXLbpJgyQ...,0.096510,1.000000,0.000000,0.333333,0.8633,0.408267
56,693134,Dune: Part Two,"[Science Fiction, Adventure]",8.140,/6izwz7rsy95ARzTR3poZ8H6c5pp.jpg,https://image.tmdb.org/t/p/w500/6izwz7rsy95ARz...,0.049604,1.000000,0.000000,0.333333,0.8140,0.384575


tesing all the module of ML pipe line


In [ ]:
#Validate your enriched dataset

# STEP 1: Validate the enriched dataset

import pandas as pd
import numpy as np
import os

DATA_PATH = "/content/movies_enriched.csv"

assert os.path.exists(DATA_PATH), (
    f"File not found: {DATA_PATH}"
)

check_df = pd.read_csv(DATA_PATH)

print("Dataset validation")
print("-" * 40)
print("Total movies:", len(check_df))
print("Total columns:", len(check_df.columns))
print("\nColumns:")
print(check_df.columns.tolist())

required_columns = [
    "tmdb_id",
    "title",
    "genres",
    "overview",
    "vote_average",
    "poster_path"
]

missing_columns = [
    col for col in required_columns
    if col not in check_df.columns
]

print("\nMissing required columns:", missing_columns)
print("Duplicate movie IDs:", check_df["tmdb_id"].duplicated().sum()
      if "tmdb_id" in check_df.columns else "N/A")
print("Missing titles:", check_df["title"].isna().sum()
      if "title" in check_df.columns else "N/A")

print("\nSample records:")
display(check_df.head(5))


Dataset validation
----------------------------------------
Total movies: 600
Total columns: 18

Columns:
['tmdb_id', 'title', 'original_title', 'original_language', 'release_date', 'overview', 'genre_ids', 'vote_average', 'vote_count', 'popularity', 'poster_path', 'adult', 'original_language_name', 'poster_url', 'genres', 'cast', 'director', 'keywords']

Missing required columns: []
Duplicate movie IDs: 0
Missing titles: 0

Sample records:


,tmdb_id,title,original_title,original_language,release_date,overview,genre_ids,vote_average,vote_count,popularity,poster_path,adult,original_language_name,poster_url,genres,cast,director,keywords
0,969681,Spider-Man: Brand New Day,Spider-Man: Brand New Day,en,2026-07-29,Fighting crime full-time as Spider-Man in a wo...,"[878, 28, 12]",8.400,4936,986.0191,/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg,False,English,https://image.tmdb.org/t/p/w500/bjiS5ipwxb9JFy...,"['Science Fiction', 'Action', 'Adventure']","['Tom Holland', 'Sadie Sink', 'Tramell Tillman...",Destin Daniel Cretton,"['mind control', 'new york city', 'hero', 'mut..."
1,1423191,Resident Evil,Resident Evil,en,2026-09-16,Medical courier Bryan unwittingly finds himsel...,"[27, 878, 12]",7.300,859,704.1066,/i7UyjfPio0VFHB9rBUZSFyhOoM8.jpg,False,English,https://image.tmdb.org/t/p/w500/i7UyjfPio0VFHB...,"['Horror', 'Science Fiction', 'Adventure']","['Austin Abrams', 'Paul Walter Hauser', 'Kali ...",Zach Cregger,"['outbreak', 'survival', 'decapitation', 'zomb..."
2,1291595,Insidious: Out of the Further,Insidious: Out of the Further,en,2026-08-19,"Gemma, a young mother raising her daughter in ...","[27, 53]",7.600,568,454.8876,/4tTrW9dXCByS5wt2pXVWb58zNjz.jpg,False,English,https://image.tmdb.org/t/p/w500/4tTrW9dXCByS5w...,"['Horror', 'Thriller']","['Amelia Eve', 'Island Austin', 'Brandon Perea...",Jacob Chase,"['supernatural', 'trauma', 'cult', 'sequel', '..."
3,1368337,The Odyssey,The Odyssey,en,2026-07-15,"Odysseus, the legendary King of Ithaca, embark...","[12, 28, 14]",8.011,4085,389.5944,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,False,English,https://image.tmdb.org/t/p/w500/5rhTDKUhPYvpdQ...,"['Adventure', 'Action', 'Fantasy']","['Matt Damon', 'Tom Holland', 'Anne Hathaway',...",Christopher Nolan,"['ship', 'trojan war', 'greek mythology', 'his..."
4,1228834,The Fix,The Fix,en,2026-09-10,Disillusioned by the end of the war in Afghani...,"[28, 53]",6.773,55,276.9325,/yopXjun3ICFfJci2ukcEzceZjUs.jpg,False,English,https://image.tmdb.org/t/p/w500/yopXjun3ICFfJc...,"['Action', 'Thriller']","['Zachary Levi', 'Liam Neeson', 'Wes Chatham',...",Guy Moshe,"['central intelligence agency (cia)', 'war vet..."


In [ ]:
#Check that recommendation model is aligned

# STEP 2: Check model dimensions and alignment

print("Movies in movies_df:", len(movies_df))
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Cosine similarity shape:", cosine_sim.shape)

assert len(movies_df) == tfidf_matrix.shape[0], (
    "Mismatch between movies_df and TF-IDF matrix. "
    "Rebuild the feature matrix."
)

assert cosine_sim.shape == (
    len(movies_df), len(movies_df)
), "Cosine similarity matrix dimensions do not match."

assert movies_df["title"].notna().all(), (
    "Some movie titles are missing."
)

print("\nModel dimensions are consistent.")


Movies in movies_df: 600
TF-IDF matrix shape: (600, 10000)
Cosine similarity shape: (600, 600)

Model dimensions are consistent.


In [ ]:
#Test recommendations using a real title

# STEP 3: Test the content-based recommender

print("Example movie titles:")
display(
    movies_df[["title", "genres"]]
    .drop_duplicates("title")
    .head(20)
)

# Replace this with a title printed above
test_title = movies_df["title"].dropna().iloc[0]

print("\nRecommendations for:", test_title)

test_recommendations = recommend_movies(
    test_title,
    top_n=10
)

display(test_recommendations)


Example movie titles:


,title,genres
0,Spider-Man: Brand New Day,"[Science Fiction, Action, Adventure]"
1,Resident Evil,"[Horror, Science Fiction, Adventure]"
2,Insidious: Out of the Further,"[Horror, Thriller]"
3,The Odyssey,"[Adventure, Action, Fantasy]"
4,The Fix,"[Action, Thriller]"
5,Runner,"[Action, Comedy]"
6,Fall 2: Deadpoint,[Thriller]
7,After Impact,"[Science Fiction, Drama, Thriller]"
8,Animals,"[Thriller, Drama]"
9,Resident Dead,[Horror]



Recommendations for: Spider-Man: Brand New Day


,tmdb_id,title,genres,overview,vote_average,poster_path,poster_url,similarity_score
0,634649,Spider-Man: No Way Home,"[Action, Adventure, Science Fiction]",Peter Parker is unmasked and no longer able to...,7.947,/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg,https://image.tmdb.org/t/p/w500/1g0dhYtq4irTY1...,0.520230
1,24428,The Avengers,"[Science Fiction, Action, Adventure]",When an unexpected enemy emerges and threatens...,8.080,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,https://image.tmdb.org/t/p/w500/RYMX2wcKCBAr24...,0.194500
2,1003596,Avengers: Doomsday,"[Science Fiction, Action, Adventure]",Beloved heroes from three distinct universes a...,0.000,/jzPwsojjFStf5lR5Nm07w2hH56G.jpg,https://image.tmdb.org/t/p/w500/jzPwsojjFStf5l...,0.183803
3,299536,Avengers: Infinity War,"[Adventure, Action, Science Fiction]",As the Avengers and their allies have continue...,8.244,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,https://image.tmdb.org/t/p/w500/7WsyChQLEftFiD...,0.174790
4,299534,Avengers: Endgame,"[Adventure, Science Fiction, Action]",After the devastating events of Avengers: Infi...,8.245,/ulzhLuWrPK07P1YkdWQLZnQh1JL.jpg,https://image.tmdb.org/t/p/w500/ulzhLuWrPK07P1...,0.151692
5,533535,Deadpool & Wolverine,"[Action, Comedy, Science Fiction]",A listless Wade Wilson toils away in civilian ...,7.585,/8cdWjvZQUExUUTzyp4t6EDMubfO.jpg,https://image.tmdb.org/t/p/w500/8cdWjvZQUExUUT...,0.142216
6,1159311,My Hero Academia: You're Next,"[Animation, Action, Adventure, Science Fiction]",In a society devastated by the effects of an a...,7.300,/tTrI6PwqzxkgO3dvQ7BEKXM7SYR.jpg,https://image.tmdb.org/t/p/w500/tTrI6PwqzxkgO3...,0.107525
7,657644,Minnal Murali,"[Action, Comedy, Fantasy]",A tailor gains special powers after being stru...,7.200,/efetKFDyptrRpoHBb103Tg3Auw5.jpg,https://image.tmdb.org/t/p/w500/efetKFDyptrRpo...,0.084746
8,1275779,Disclosure Day,"[Science Fiction, Thriller]",A cybersecurity expert becomes a whistleblower...,7.400,/AnJ8IQJI23hNpYXVNaythu061Ru.jpg,https://image.tmdb.org/t/p/w500/AnJ8IQJI23hNpY...,0.083266
9,149,Akira,"[Animation, Science Fiction, Action]",A secret military project endangers Neo-Tokyo ...,7.938,/neZ0ykEsPqxamsX6o5QNUFILQrz.jpg,https://image.tmdb.org/t/p/w500/neZ0ykEsPqxams...,0.082073


In [ ]:
#Test final personalized recommendations

# STEP 4: Inspect final personalized recommendations

# Verify required variables are in the global namespace
required_globals = ["user_history", "selected_mood", "selected_context"]
missing_globals = [name for name in required_globals if name not in globals()]
assert not missing_globals, f"Run the earlier setup cells first. Missing global variables: {missing_globals}"

# Verify score columns exist in movies_df rather than as global variables
required_columns = [
    "preference_score",
    "mood_score",
    "context_score",
    "final_score"
]

missing_columns = [
    col for col in required_columns
    if col not in movies_df.columns
]

assert not missing_columns, (
    f"Run the scoring cells first (3fQGrWynXF1o, jZoP2o_rXaxl, and EuBB4zw9XfxA). "
    f"Missing columns in movies_df: {missing_columns}"
)

result_columns = [
    col for col in [
        "tmdb_id",
        "title",
        "genres",
        "vote_average",
        "content_score",
        "preference_score",
        "mood_score",
        "context_score",
        "final_score",
        "poster_path"
    ]
    if col in movies_df.columns
]

display(
    movies_df.sort_values("final_score", ascending=False)
    [result_columns]
    .head(10)
)

,tmdb_id,title,genres,vote_average,content_score,preference_score,mood_score,context_score,final_score,poster_path
0,969681,Spider-Man: Brand New Day,"[Science Fiction, Action, Adventure]",8.400,1.000000,2.000000,0.000000,0.666667,0.800667,/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg
32,299534,Avengers: Endgame,"[Adventure, Science Fiction, Action]",8.245,1.000000,2.000000,0.000000,0.666667,0.799117,/ulzhLuWrPK07P1YkdWQLZnQh1JL.jpg
49,634649,Spider-Man: No Way Home,"[Action, Adventure, Science Fiction]",7.947,0.520230,2.000000,0.000000,0.666667,0.604228,/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg
50,24428,The Avengers,"[Science Fiction, Action, Adventure]",8.080,0.453151,2.000000,0.000000,0.666667,0.578727,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg
36,299536,Avengers: Infinity War,"[Adventure, Action, Science Fiction]",8.244,0.416990,2.000000,0.000000,0.666667,0.565903,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg
33,1003596,Avengers: Doomsday,"[Science Fiction, Action, Adventure]",0.000,0.428079,2.000000,0.000000,0.666667,0.487898,/jzPwsojjFStf5lR5Nm07w2hH56G.jpg
588,1159311,My Hero Academia: You're Next,"[Animation, Action, Adventure, Science Fiction]",7.300,0.107525,1.500000,0.333333,0.666667,0.420176,/tTrI6PwqzxkgO3dvQ7BEKXM7SYR.jpg
55,1228710,The Mandalorian and Grogu,"[Action, Adventure, Science Fiction]",7.500,0.062401,2.000000,0.000000,0.666667,0.416627,/uwMKWjcNID0D9jjplsjkQS2OrB4.jpg
59,533535,Deadpool & Wolverine,"[Action, Comedy, Science Fiction]",7.585,0.142216,1.333333,0.333333,0.666667,0.416070,/8cdWjvZQUExUUTzyp4t6EDMubfO.jpg
14,1148365,Matchbox the Movie,"[Action, Comedy, Adventure]",7.857,0.038222,1.333333,0.333333,1.000000,0.410525,/lFKA4GMz36dImEy3DQPdPF7G4V9.jpg


In [ ]:

# Save model artifacts

import joblib
from scipy.sparse import save_npz
import json
import os

ARTIFACT_DIR = "/content/movie_recommender_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Save processed movie data
movies_df.to_csv(
    f"{ARTIFACT_DIR}/movies_featured.csv",
    index=False
)

movies_df.to_json(
    f"{ARTIFACT_DIR}/movies_featured.json",
    orient="records",
    force_ascii=False,
    indent=2
)

# Save the fitted TF-IDF vectorizer
joblib.dump(
    tfidf,
    f"{ARTIFACT_DIR}/tfidf_vectorizer.pkl"
)

# Save the sparse cosine similarity matrix
save_npz(
    f"{ARTIFACT_DIR}/cosine_similarity.npz",
    cosine_sim
)

# Save the initial scoring configuration
scoring_config = {
    "content_weight": 0.40,
    "preference_weight": 0.25,
    "mood_weight": 0.15,
    "context_weight": 0.10,
    "rating_weight": 0.10
}

with open(
    f"{ARTIFACT_DIR}/scoring_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(scoring_config, f, indent=2)

print("Saved artifacts:")
for filename in sorted(os.listdir(ARTIFACT_DIR)):
    print("-", filename)


Saved artifacts:
- cosine_similarity.npz
- movies_featured.csv
- movies_featured.json
- scoring_config.json
- tfidf_vectorizer.pkl
